# MAT 167 Final Project


In [50]:
# Import dependecies and libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
import random
import string


### Step 1: Turn hyperlink of nodes into Google Matrix G and check all row sums equal to 1.

In [51]:
import numpy as np

n = 12
nodes = list('ABCDEFGHIJKL')

edges = [
    ('A','I'), ('A','B'), ('A','D'),
    ('B','E'), ('B','D'),
    ('C','B'), ('C','D'),
    # D has no outgoing edges (dangling)
    ('E','G'),
    ('F','B'), ('F','C'), ('F','E'),
    ('G','J'), ('G','E'),
    ('H','F'), ('H','G'), ('H','B'),
    ('I','G'), ('I','J'),
    ('J','K'), ('J','H'),
    ('K','H'),
    ('L','H'), ('L','J'),
]

# Build raw adjacency matrix
idx = {node: i for i, node in enumerate(nodes)}
G = np.zeros((n, n))

for src, dst in edges:
    G[idx[src], idx[dst]] = 1

# Identify dangling nodes (nodes with no outlinks)
dangling_nodes = []
for i, node in enumerate(nodes):
    row_sum = G[i].sum()
    if row_sum == 0:
        dangling_nodes.append(node)
        print(f"Node {node} is dangling (no outlinks) - will set row to 1/{n}")

print(f"\nDangling nodes found: {dangling_nodes}")

# Normalize each row to sum to 1 (row stochastic)
# Fix dangling nodes by setting their row to 1/n everywhere
for i in range(n):
    row_sum = G[i].sum()
    if row_sum == 0:
        G[i] = 1.0/n  # Dangling node: equal probability to all nodes
        print(f"Fixed row {nodes[i]}: set to 1/{n} = {1.0/n:.6f} everywhere")
    else:
        G[i] /= row_sum

# Verify row D specifically
d_idx = idx['D']
print(f"\nRow D in final matrix G:")
print(f"Row D = {np.round(G[d_idx], 6)}")
print(f"Row D sum = {G[d_idx].sum():.6f} (should be 1.0)")
print(f"All entries equal? {np.allclose(G[d_idx], 1.0/n)}")

df_G = pd.DataFrame(G)
df_G.columns = list(string.ascii_uppercase[:n])
df_G.index = list(string.ascii_uppercase[:n])
display(df_G)

# Check Row Sums
print("\nRow sums:", np.round(G.sum(axis=1), 4))

Node D is dangling (no outlinks) - will set row to 1/12

Dangling nodes found: ['D']
Fixed row D: set to 1/12 = 0.083333 everywhere

Row D in final matrix G:
Row D = [0.083333 0.083333 0.083333 0.083333 0.083333 0.083333 0.083333 0.083333
 0.083333 0.083333 0.083333 0.083333]
Row D sum = 1.000000 (should be 1.0)
All entries equal? True


,A,B,C,D,E,F,G,H,I,J,K,L
A,0.000000,0.333333,0.000000,0.333333,0.000000,0.000000,0.000000,0.000000,0.333333,0.000000,0.000000,0.000000
B,0.000000,0.000000,0.000000,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
C,0.000000,0.500000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
D,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333
E,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
F,0.000000,0.333333,0.333333,0.000000,0.333333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
G,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000
H,0.000000,0.333333,0.000000,0.000000,0.000000,0.333333,0.333333,0.000000,0.000000,0.000000,0.000000,0.000000
I,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.500000,0.000000,0.000000
J,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.500000,0.000000



Row sums: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### Step 2: PageRank vs Eigendecomposition


In [52]:
# Compute eigenvalues and eigenvectors of G^T
eigenvals, eigenvecs = np.linalg.eig(G.T)

# Find the index where eigenvalue is closest to 1
idx_one = np.argmin(np.abs(eigenvals - 1.0))

# Get the corresponding eigenvector
pi_raw = eigenvecs[:, idx_one]

# Normalize to make it a probability distribution (sum to 1)
# Take real part and ensure positive
pi = np.real(pi_raw)
pi = np.abs(pi)
pi = pi / pi.sum()  # Normalize so sum = 1

print(f"Eigenvalue found: {eigenvals[idx_one]:.6f}")
print(f"\nPageRank vector π (normalized):")
print(np.round(pi, 4))
print(f"\nSum of π: {pi.sum():.6f}")

# Display as DataFrame with node labels
df_pi = pd.DataFrame({
    'Node': nodes,
    'PageRank': np.round(pi, 6)
})
df_pi = df_pi.sort_values(by="PageRank", ascending=False)
display(df_pi)

Eigenvalue found: 1.000000+0.000000j

PageRank vector π (normalized):
[0.0049 0.0829 0.0224 0.0593 0.1836 0.0525 0.2394 0.1428 0.0066 0.1304
 0.0701 0.0049]

Sum of π: 1.000000


,Node,PageRank
6,G,0.239440
4,E,0.183621
7,H,0.142766
9,J,0.130420
1,B,0.082906
10,K,0.070149
3,D,0.059261
5,F,0.052527
2,C,0.022448
8,I,0.006585
